# GPS Outlier Detection & Cleaning

This notebook reads the multi-participant GPS Excel file and applies:

- `flag_range` → removes invalid/zero coordinates
- `flag_iqr` → removes participant-specific latitude/longitude IQR outliers
- `flag_speed` → diagnostic only
- `flag_speed_accuracy` → diagnostic only

Only coordinate outliers are removed from the final cleaned CSV.

In [ ]:
# STEP 1 — Imports
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

In [ ]:
# STEP 2 — Read your Excel file
# Keep the Excel file in the SAME folder as this notebook.

INPUT_FILE = "Location_multi_participants (3).xlsx"

if not Path(INPUT_FILE).exists():
    raise FileNotFoundError(
        f"Could not find {INPUT_FILE}. Put the Excel file in the same folder as this notebook."
    )

df = pd.read_excel(INPUT_FILE)

print(f"File loaded: {INPUT_FILE}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nOriginal columns:")
print(df.columns.tolist())

In [ ]:
# STEP 3 — Standardize column names
def normalize_col(name):
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

df.columns = [normalize_col(c) for c in df.columns]

# Common alternative names
aliases = {
    "participant": "participant_id",
    "participantid": "participant_id",
    "lat": "latitude",
    "lng": "longitude",
    "lon": "longitude",
    "long": "longitude",
    "measurementid": "measurement_id",
    "speedaccuracy": "speed_accuracy",
    "speed_acc": "speed_accuracy",
}

df.rename(
    columns={c: aliases[c] for c in df.columns if c in aliases},
    inplace=True
)

required = [
    "participant_id",
    "latitude",
    "longitude",
    "speed",
    "speed_accuracy"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        "Missing required columns: "
        + str(missing)
        + "\n\nAvailable columns:\n"
        + str(df.columns.tolist())
    )

# Create measurement_id if the source file does not contain it
if "measurement_id" not in df.columns:
    df["measurement_id"] = np.arange(1, len(df) + 1)

print("Required columns found.")
print("Participants:", df["participant_id"].nunique())

In [ ]:
# STEP 4 — Convert relevant columns to numeric
for col in ["latitude", "longitude", "speed", "speed_accuracy"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[["latitude", "longitude", "speed", "speed_accuracy"]].isna().sum())

In [ ]:
# STEP 5 — Thresholds
IQR_MULTIPLIER = 1.5

# 55.56 m/s = 200 km/h
MAX_SPEED_MS = 55.56

# Maximum acceptable reported speed accuracy in metres
MAX_SPEED_ACCURACY = 100.0

print("IQR multiplier:", IQR_MULTIPLIER)
print("Maximum speed:", MAX_SPEED_MS, "m/s")
print("Maximum speed accuracy:", MAX_SPEED_ACCURACY, "m")

In [ ]:
# STEP 6 — Helper functions

def flag_invalid_range(data):
    return (
        data["latitude"].isna()
        | data["longitude"].isna()
        | (data["latitude"] < -90)
        | (data["latitude"] > 90)
        | (data["longitude"] < -180)
        | (data["longitude"] > 180)
        | (data["latitude"] == 0)
        | (data["longitude"] == 0)
    )


def flag_iqr_outliers(group, multiplier=IQR_MULTIPLIER):
    outlier = pd.Series(False, index=group.index)

    for col in ["latitude", "longitude"]:
        values = group[col].dropna()

        # IQR is not meaningful with too few observations
        if len(values) < 4:
            continue

        Q1 = values.quantile(0.25)
        Q3 = values.quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR

        outlier |= (
            (group[col] < lower)
            | (group[col] > upper)
        )

    return outlier


def flag_speed_outliers(data, max_speed_ms=MAX_SPEED_MS):
    return (
        data["speed"].isna()
        | (data["speed"] == -1)
        | (data["speed"] > max_speed_ms)
    )


def flag_speed_accuracy_outliers(
    data,
    max_accuracy_ms=MAX_SPEED_ACCURACY
):
    return (
        data["speed_accuracy"].isna()
        | (data["speed_accuracy"] == -1)
        | (data["speed_accuracy"] > max_accuracy_ms)
    )

In [ ]:
# STEP 7 — Apply all four flags

df["flag_range"] = flag_invalid_range(df)

# Participant-specific IQR
iqr_flags = (
    df.groupby("participant_id", sort=False)
      .apply(
          lambda g: flag_iqr_outliers(g),
          include_groups=False
      )
)

# Convert the groupby result back to the original row index
if isinstance(iqr_flags.index, pd.MultiIndex):
    iqr_flags.index = iqr_flags.index.get_level_values(-1)

df["flag_iqr"] = (
    iqr_flags
    .reindex(df.index)
    .fillna(False)
    .astype(bool)
)

df["flag_speed"] = flag_speed_outliers(df)

df["flag_speed_accuracy"] = flag_speed_accuracy_outliers(df)

# ONLY coordinate flags remove rows
location_flag_cols = [
    "flag_range",
    "flag_iqr"
]

diagnostic_flag_cols = [
    "flag_speed",
    "flag_speed_accuracy"
]

all_flag_cols = (
    location_flag_cols
    + diagnostic_flag_cols
)

df["is_location_outlier"] = (
    df[location_flag_cols].any(axis=1)
)

df["is_any_flag"] = (
    df[all_flag_cols].any(axis=1)
)

df["methods_flagged"] = (
    df[all_flag_cols].sum(axis=1)
)

df["is_overlap"] = (
    df["methods_flagged"] >= 2
)

# Backward-compatible name
df["is_outlier"] = df["is_location_outlier"]

print("Flags applied.")
print(
    f"Location outliers to remove: "
    f"{df['is_location_outlier'].sum():,}"
)
print(
    f"Rows with any flag: "
    f"{df['is_any_flag'].sum():,}"
)

In [ ]:
# STEP 8 — Global outlier breakdown

print("=" * 60)
print("GLOBAL OUTLIER BREAKDOWN")
print("=" * 60)

print(
    f"Invalid coordinate range : "
    f"{df['flag_range'].sum():,}"
)

print(
    f"Coordinate IQR           : "
    f"{df['flag_iqr'].sum():,}"
)

print(
    f"LOCATION rows removed    : "
    f"{df['is_location_outlier'].sum():,} / "
    f"{len(df):,} "
    f"({df['is_location_outlier'].mean()*100:.2f}%)"
)

print(
    f"Location rows retained   : "
    f"{(~df['is_location_outlier']).sum():,}"
)

print("\nDiagnostic only — DO NOT remove coordinates:")

print(
    f"Speed sentinel/high      : "
    f"{df['flag_speed'].sum():,}"
)

print(
    f"Poor speed accuracy      : "
    f"{df['flag_speed_accuracy'].sum():,}"
)

print(
    f"Any flag                 : "
    f"{df['is_any_flag'].sum():,}"
)

In [ ]:
# STEP 9 — Pairwise overlap matrix

flag_labels = {
    "flag_range": "Range",
    "flag_iqr": "IQR",
    "flag_speed": "Speed",
    "flag_speed_accuracy": "SpeedAcc"
}

overlap_matrix = pd.DataFrame(
    index=flag_labels.values(),
    columns=flag_labels.values(),
    dtype=int
)

for c1, l1 in flag_labels.items():
    for c2, l2 in flag_labels.items():
        overlap_matrix.loc[l1, l2] = int(
            (df[c1] & df[c2]).sum()
        )

display(overlap_matrix)

In [ ]:
# STEP 10 — Participant-level summary

outlier_summary = (
    df.groupby("participant_id")
      .agg(
          total_points=("measurement_id", "count"),
          location_outlier_count=("is_location_outlier", "sum"),
          range_outliers=("flag_range", "sum"),
          iqr_outliers=("flag_iqr", "sum"),
          speed_flags=("flag_speed", "sum"),
          speed_accuracy_flags=("flag_speed_accuracy", "sum"),
          overlap_points=("is_overlap", "sum"),
      )
      .reset_index()
)

outlier_summary["location_outlier_pct"] = (
    outlier_summary["location_outlier_count"]
    / outlier_summary["total_points"]
    * 100
).round(2)

outlier_summary["clean_points"] = (
    outlier_summary["total_points"]
    - outlier_summary["location_outlier_count"]
)

display(outlier_summary)

In [ ]:
# STEP 11 — Participant-wise overlap detail

pairs = [
    ("flag_range", "flag_iqr", "Range & IQR"),
    ("flag_range", "flag_speed", "Range & Speed"),
    ("flag_range", "flag_speed_accuracy", "Range & SpeedAcc"),
    ("flag_iqr", "flag_speed", "IQR & Speed"),
    ("flag_iqr", "flag_speed_accuracy", "IQR & SpeedAcc"),
    ("flag_speed", "flag_speed_accuracy", "Speed & SpeedAcc"),
]

overlap_records = []

for pid, group in df.groupby("participant_id"):
    row = {"participant_id": pid}

    for c1, c2, label in pairs:
        row[label] = int(
            (group[c1] & group[c2]).sum()
        )

    overlap_records.append(row)

overlap_detail = pd.DataFrame(overlap_records)

display(overlap_detail)

In [ ]:
# STEP 12 — Create the cleaned dataset

# Audit file: original rows + all flags
df.to_csv(
    "gps_with_outlier_flags.csv",
    index=False
)

# TRUE cleaning:
# remove ONLY coordinate outliers
clean = df.loc[
    ~df["is_location_outlier"]
].copy()

# Final coordinate validity guard
valid_coord = (
    clean["latitude"].notna()
    & clean["longitude"].notna()
    & clean["latitude"].between(-90, 90)
    & clean["longitude"].between(-180, 180)
    & clean["latitude"].ne(0)
    & clean["longitude"].ne(0)
)

clean = clean.loc[valid_coord].copy()

# Save outputs
clean.to_csv(
    "gps_outliers_removed.csv",
    index=False
)

outlier_summary.to_csv(
    "outlier_summary_per_participant.csv",
    index=False
)

overlap_detail.to_csv(
    "overlap_detail_per_participant.csv",
    index=False
)

print(f"Original rows : {len(df):,}")
print(f"Clean rows    : {len(clean):,}")
print(f"Rows removed  : {len(df) - len(clean):,}")
print(
    f"Participants in source : "
    f"{df['participant_id'].nunique()}"
)
print(
    f"Participants with >=1 clean GPS row: "
    f"{clean['participant_id'].nunique()}"
)

In [ ]:
# STEP 13 — Final verification

print("FINAL CHECKS")
print("=" * 60)

print(
    "Missing latitude:",
    clean["latitude"].isna().sum()
)

print(
    "Missing longitude:",
    clean["longitude"].isna().sum()
)

print(
    "Zero latitude:",
    (clean["latitude"] == 0).sum()
)

print(
    "Zero longitude:",
    (clean["longitude"] == 0).sum()
)

print(
    "Invalid latitude:",
    (~clean["latitude"].between(-90, 90)).sum()
)

print(
    "Invalid longitude:",
    (~clean["longitude"].between(-180, 180)).sum()
)

print("\nOutput files created:")
for filename in [
    "gps_with_outlier_flags.csv",
    "gps_outliers_removed.csv",
    "outlier_summary_per_participant.csv",
    "overlap_detail_per_participant.csv"
]:
    print(" -", filename)